### Dynamic Array


In [3]:
from __future__ import annotations
from typing import Generic, TypeVar, Iterable, Iterator, Optional

T = TypeVar("T")


class DynamicArray(Generic[T]):
    __slots__ = ("_size", "_capacity", "_data")

    DEFAULT_CAPACITY = 4
    GROWTH_FACTOR = 2
    SHRINK_THRESHOLD = 0.25

    def __init__(self, iterable: Optional[Iterable[T]] = None) -> None:
        self._size = 0
        self._capacity = self.DEFAULT_CAPACITY
        self._data = self._make_array(self._capacity)

        if iterable is not None:
            for item in iterable:
                self.append(item)

    def __len__(self) -> int:
        return self._size

    def __getitem__(self, index: int) -> T:
        index = self._normalize_index(index)
        return self._data[index]

    def __setitem__(self, index: int, value: T) -> None:
        index = self._normalize_index(index)
        self._data[index] = value

    def __iter__(self) -> Iterator[T]:
        for i in range(self._size):
            yield self._data[i]

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}({list(self)})"

    def append(self, value: T) -> None:
        if self._size == self._capacity:
            self._resize(self._capacity * self.GROWTH_FACTOR)

        self._data[self._size] = value
        self._size += 1

    def pop(self) -> T:
        if self._size == 0:
            raise IndexError("pop from empty DynamicArray")

        value = self._data[self._size - 1]
        self._data[self._size - 1] = None
        self._size -= 1

        if (
                self._capacity > self.DEFAULT_CAPACITY
                and self._size < self._capacity * self.SHRINK_THRESHOLD
        ):
            new_capacity = max(
                self.DEFAULT_CAPACITY,
                self._capacity // self.GROWTH_FACTOR,
            )
            self._resize(new_capacity)

        return value  # type: ignore

    def insert(self, index: int, value: T) -> None:
        if index < 0:
            index += self._size
        if index < 0:
            index = 0
        if index > self._size:
            index = self._size

        if self._size == self._capacity:
            self._resize(self._capacity * self.GROWTH_FACTOR)

        for i in range(self._size, index, -1):
            self._data[i] = self._data[i - 1]

        self._data[index] = value
        self._size += 1

    def remove(self, value: T) -> None:
        for i in range(self._size):
            if self._data[i] == value:
                self._delete_at_index(i)
                return
        raise ValueError(f"{value} not found")

    def clear(self) -> None:
        self._data = self._make_array(self.DEFAULT_CAPACITY)
        self._size = 0
        self._capacity = self.DEFAULT_CAPACITY

    def capacity(self) -> int:
        return self._capacity

    def is_empty(self) -> bool:
        return self._size == 0

    def _delete_at_index(self, index: int) -> None:
        for i in range(index, self._size - 1):
            self._data[i] = self._data[i + 1]

        self._data[self._size - 1] = None
        self._size -= 1

        if (
                self._capacity > self.DEFAULT_CAPACITY
                and self._size < self._capacity * self.SHRINK_THRESHOLD
        ):
            new_capacity = max(
                self.DEFAULT_CAPACITY,
                self._capacity // self.GROWTH_FACTOR,
            )
            self._resize(new_capacity)

    def _resize(self, new_capacity: int) -> None:
        new_data = self._make_array(new_capacity)
        for i in range(self._size):
            new_data[i] = self._data[i]

        self._data = new_data
        self._capacity = new_capacity

    def _normalize_index(self, index: int) -> int:
        if not isinstance(index, int):
            raise TypeError("index must be an integer")

        if index < 0:
            index += self._size

        if index < 0 or index >= self._size:
            raise IndexError("index out of range")

        return index

    @staticmethod
    def _make_array(capacity: int):
        return [None] * capacity


In [4]:
arr = DynamicArray([1,2,3])
arr.append(4)
len(arr)


4


In [11]:
B = [1,2]
A = [1, 2, B]
A


[1, 2, [1, 2]]


### Storing High Scores for a Game


In [6]:
class Entry:
    def __init__(self, name, score):
        self._name = name
        self._score = score

    def get_score(self):
        return self._score

    def get_name(self):
        return self._name

    def __str__(self):
        return '({0}, {1})'.format(self._name, self._score)

class Scoreboard:
    def __init__(self, capacity=10):
        self._board = [None] * capacity
        self._n = 0

    def __getitem__(self, k):
        return self._board[k]

    def __str__(self):
        return '\n'.join(str(self._board[j]) for j in range(self._n))

    def add(self, entry):
        score = entry.get_score()

        is_good = self._n < len(self._board) or self._board[-1].get_score() < score

        if is_good:
            if self._n < len(self._board):
                self._n += 1

            j = self._n - 1
            # we had already checked that score > self._board[j].get_score() in line 29, now start checking with j-1
            while j > 0 and self._board[j-1].get_score() < score:
                self._board[j] = self._board[j-1]
                j -= 1

            self._board[j] = entry


### Sorting a Sequence
**Insertion Sort**


In [7]:
def insertion_sort(A):
    for k in range(len(A)):
        cur = A[k]
        j = k
        while j>0 and cur<A[j-1]:
            A[j] = A[j-1]
            j -= 1
        A[j] = cur
    return A

A = [3, 5, 7, 4, 2, 9, 5]
A = insertion_sort(A)
print(A)


[2, 3, 4, 5, 5, 7, 9]


### Simple Cryptography


In [8]:
class CaesarCipher:
    def __init__(self, shift: int):
        self.shift = shift % 26

    def encrypt(self, text: str) -> str:
        return self._transform(text, self.shift)

    def decrypt(self, text: str) -> str:
        return self._transform(text, -self.shift)

    def _transform(self, text: str, shift: int) -> str:
        result = []

        for char in text:
            if char.isupper():
                base = ord('A')
                result.append(chr((ord(char) - base + shift) % 26 + base))
            elif char.islower():
                base = ord('a')
                result.append(chr((ord(char) - base + shift) % 26 + base))
            else:
                result.append(char)

        return ''.join(result)


In [9]:
cipher = CaesarCipher(3)

encrypted = cipher.encrypt("Hello, World!")
print(encrypted)  # Khoor, Zruog!

decrypted = cipher.decrypt(encrypted)
print(decrypted)  # Hello, World!


Khoor, Zruog!
Hello, World!


### Tic Tac Toe


In [10]:
board = [
    [" ", " ", " "],
    [" ", " ", " "],
    [" ", " ", " "]
]


def print_board():
    print()
    for i in range(3):
        print(" " + board[i][0] + " | " + board[i][1] + " | " + board[i][2])
        if i < 2:
            print("---+---+---")
    print()


def check_winner(player):
    # Check rows
    for row in board:
        if row[0] == player and row[1] == player and row[2] == player:
            return True

    # Check columns
    for col in range(3):
        if board[0][col] == player and board[1][col] == player and board[2][col] == player:
            return True

    # Check diagonals
    if board[0][0] == player and board[1][1] == player and board[2][2] == player:
        return True

    if board[0][2] == player and board[1][1] == player and board[2][0] == player:
        return True

    return False


def is_board_full():
    for row in board:
        for cell in row:
            if cell == " ":
                return False
    return True


def make_move(player):
    while True:
        try:
            row = int(input(f"Player {player}, enter row number (1-3): ")) - 1
            col = int(input(f"Player {player}, enter column number (1-3): ")) - 1

            if row < 0 or row > 2 or col < 0 or col > 2:
                print("Invalid position. Row and column must be between 1 and 3.")
                continue

            if board[row][col] != " ":
                print("That position is already taken. Try again.")
                continue

            board[row][col] = player
            break

        except ValueError:
            print("Invalid input. Please enter numbers only.")


def play_game():
    current_player = "X"

    print("Tic Tac Toe")
    print("Player X goes first.")

    while True:
        print_board()
        make_move(current_player)

        if check_winner(current_player):
            print_board()
            print(f"Player {current_player} wins!")
            break

        if is_board_full():
            print_board()
            print("The game is a draw.")
            break

        if current_player == "X":
            current_player = "O"
        else:
            current_player = "X"


play_game()


Tic Tac Toe
Player X goes first.
   |   |
---+---+---
   |   |
---+---+---
   |   |


StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.